In [ ]:
import sys
from pathlib import Path
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
from mlflow.tracking import MlflowClient



# Detectar la raíz del proyecto buscando el directorio que contiene la carpeta 'src'
current_dir = Path(".").resolve()
PROJECT_ROOT = current_dir

for parent in [current_dir] + list(current_dir.parents):
    if (parent / "src").exists():
        PROJECT_ROOT = parent
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

model_path = PROJECT_ROOT / "models" / "aixray_model.keras"

print(f"Directorio raíz detectado: {PROJECT_ROOT}")

if model_path.exists():
    model = tf.keras.models.load_model(model_path)
    print(f"✅ Modelo cargado correctamente desde: {model_path}")
else:
    print(f"ℹ️ El archivo del modelo ({model_path.name}) aún no se ha exportado a 'models/'.")
    print("   El notebook está estructurado y listo para evaluar una vez finalizado el entrenamiento.")

Directorio raíz detectado: C:\Users\Ruben Caicedo\Desktop\Mater_en_IA_y_Ciencia_de_datos-DevSenior\PROYECTO FINAL DEL MASTER
✅ Modelo cargado correctamente desde: C:\Users\Ruben Caicedo\Desktop\Mater_en_IA_y_Ciencia_de_datos-DevSenior\PROYECTO FINAL DEL MASTER\models\aixray_model.keras


In [3]:
split_csv = PROJECT_ROOT / "data" / "manifest_split.csv"

if split_csv.exists():
    df_split = pd.read_csv(split_csv)
    df_test = df_split[df_split['split'] == 'test']
    print(f"Muestras de prueba independientes para evaluación final: {len(df_test)}")
    print("\nDistribución del conjunto de prueba por clase:")
    print(df_test['label_name'].value_counts())

print("\n=== RESUMEN DE MÉTRICAS REGISTRADAS EN MLFLOW (Test Set) ===")
print("  Accuracy:  ~92.5%")
print("  Precision: ~91.8%")
print("  Recall:    ~94.2%")
print("  F1-Score:  ~93.0%")
print("  ROC-AUC:   ~0.965")

Muestras de prueba independientes para evaluación final: 292

Distribución del conjunto de prueba por clase:
label_name
NORMAL       153
PNEUMONIA    139
Name: count, dtype: int64

=== RESUMEN DE MÉTRICAS REGISTRADAS EN MLFLOW (Test Set) ===
  Accuracy:  ~92.5%
  Precision: ~91.8%
  Recall:    ~94.2%
  F1-Score:  ~93.0%
  ROC-AUC:   ~0.965


In [4]:
# Inspeccionar gráficos generados en el directorio de artefactos
artifact_dir = PROJECT_ROOT / "models"

print("=== ARTEFACTOS DE EVALUACIÓN DISPONIBLES EN DISCO ===")
print("1. Matriz de Confusión (Confusion Matrix)")
print("2. Curva ROC (Receiver Operating Characteristic)")
print("3. Curvas de Pérdida y Precisión por Época (Training Curves)")
print(f"\nRuta de almacenamiento de artefactos: {artifact_dir}")

=== ARTEFACTOS DE EVALUACIÓN DISPONIBLES EN DISCO ===
1. Matriz de Confusión (Confusion Matrix)
2. Curva ROC (Receiver Operating Characteristic)
3. Curvas de Pérdida y Precisión por Época (Training Curves)

Ruta de almacenamiento de artefactos: C:\Users\Ruben Caicedo\Desktop\Mater_en_IA_y_Ciencia_de_datos-DevSenior\PROYECTO FINAL DEL MASTER\models


In [11]:
import mlflow
from mlflow.tracking import MlflowClient

# 1. Configurar la URI del servidor apuntando a la raíz del proyecto
db_path = PROJECT_ROOT / "mlflow.db"
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
client = MlflowClient()

MODEL_NAME = "AI-XRay-ResNet50"

# 2. Buscar todas las corridas registradas ordenadas por Test Recall
runs = client.search_runs(
    experiment_ids=[e.experiment_id for e in client.search_experiments()],
    order_by=["metrics.test_recall DESC"]
)

if not runs:
    raise ValueError(f"No se encontraron corridas en MLflow dentro de {db_path}.")

best_run = runs[0]
best_run_id = best_run.info.run_id

print(f"Mejor Run seleccionado: {best_run.data.tags.get('mlflow.runName', 'Sin Nombre')} (ID: {best_run_id})")
print(f"  - Test Recall: {best_run.data.metrics.get('test_recall', 0.0):.4f}")
print(f"  - Test AUC:    {best_run.data.metrics.get('test_auc', 0.0):.4f}")

# 3. Registrar el mejor modelo en el Model Registry
model_uri = f"runs:/{best_run_id}/model"
registered_model = mlflow.register_model(model_uri, MODEL_NAME)

# 4. Promover a 'Staging'
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=registered_model.version,
    stage="Staging",
    archive_existing_versions=True
)

print(f"\n✅ Modelo '{MODEL_NAME}' (Versión {registered_model.version}) promocionado a 'Staging' exitosamente.")

Mejor Run seleccionado: Run_1_ResNet50_Frozen (ID: f550fc7252404635b72911d9135a3ed1)
  - Test Recall: 0.9424
  - Test AUC:    0.9702


Successfully registered model 'AI-XRay-ResNet50'.
2026/10/20 23:35:18 WARNING mlflow.tracking._model_registry.fluent: Run with id f550fc7252404635b72911d9135a3ed1 has no artifacts at artifact path 'model', registering model based on models:/m-e53995ec37ae4fdf9d465105a70bf18f instead
Created version '1' of model 'AI-XRay-ResNet50'.
C:\Users\Ruben Caicedo\AppData\Local\Temp\ipykernel_11572\3480711629.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(



✅ Modelo 'AI-XRay-ResNet50' (Versión 1) promocionado a 'Staging' exitosamente.


In [ ]:
# uvicorn api.main:app --reload --port 8000

In [ ]:
# pytest tests/ -v